# Protein Sequence Tokenization Stretegy Comparative Analysis

This notebook implements the phase-2 Tokenization of animo acids and compare strategies based on Vocab size and

<strong>_out of scope:_</strong>

- compare strategies based on Generative and Classification tasks


In [1]:
import os
import sys
import gc
import json
import time
import traceback
import resource
from pathlib import Path

from collections import Counter
from itertools import chain

import math

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from IPython.display import display
from tqdm.auto import tqdm

from util.file_utils import ensure_directories, iter_dataset, parquet_to_txt
from util.text_similarity import rank_texts
from util.seq_util import inspect_residue_distribution
from tokenizer_module import create_tokenizer
import sentencepiece as spm

In [2]:
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


DATA_PATH = REPO_ROOT / "data/interim/protein_eda/protein_corpus.parquet"
OUTPUT_DIR = REPO_ROOT / "data/interim/protein_tokenization"
ARTIFACT_OUTPUT_DIR = OUTPUT_DIR / "artifacts"

MAX_SEQ_LENGTH = 512
PARSE_BATCH_SIZE = 2**16
# TOKENIZATION_STRATEGIES = ["Unigram", "BPE", "WordPiece", "words", "pairs", "k-mers", "SentencePiece"]
# TOKENIZATION_STRATEGIES = ["BPE"]

MIN_FREQUENCIES = [5]
KMER_SIZES = [3, 5, 6, 7, 9]
VOCAB_SIZES = [8_000]
RARE_RESIDUE_POLICY = "keep"  #  None replace_with_unk replace_with_nn
SAMPLE_MODE = None  # None  #500

SAVE_ARTIFACTS = True
SAVE_FIGURES = True

TOKENIZER_CONFIGS = (
    [{"name": "words", "requires_training": False}]
    + [{"name": "Unigram", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "BPE", "requires_training": True, "vocab_size": vs, "min_frequency": mf} for vs in VOCAB_SIZES for mf in MIN_FREQUENCIES]
    + [{"name": "WordPiece", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "pairs", "requires_training": False, "pair_mode": "sliding"}]
    + [{"name": "k-mers", "requires_training": False, "k": k, "mode": "sliding"} for k in KMER_SIZES]
    + [{"name": "k-mers", "requires_training": False, "k": k, "mode": "non-overlapping"} for k in KMER_SIZES]
    + [{"name": "SentencePiece", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "TFIDF", "requires_training": True}]
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
ensure_directories(ARTIFACT_OUTPUT_DIR, OUTPUT_DIR)
print("repo root:", REPO_ROOT)

repo root: /home/ubuntu/projects/biodata/DNA-BERT


## Parse Corpus

Parse every available `protein.faa` file, preserve provenance, and surface malformed records or headers.


In [1]:
protein_dataset = ds.dataset(str(DATA_PATH), format="parquet")

dataset_schema = pd.DataFrame(
    {
        "column": protein_dataset.schema.names,
        "dtype": [str(t) for t in protein_dataset.schema.types],
    }
)
records_total_count = protein_dataset.count_rows()
print(f"Records: {records_total_count:,}")

if SAMPLE_MODE is not None:
    print(f"Taking a sample of {SAMPLE_MODE} records")
    sample_table = protein_dataset.head(SAMPLE_MODE, columns=["sequence"])
    protein_dataset = ds.dataset(sample_table)
    records_total_count = protein_dataset.count_rows()

    if SAVE_ARTIFACTS:
        pq.write_table(
            protein_dataset.to_table(),
            ARTIFACT_OUTPUT_DIR / "sampled_protein_dataset.parquet",
            compression="snappy",
        )

display(dataset_schema)

NameError: name 'ds' is not defined

In [ ]:
TOTAL_RECORDS = records_total_count
TOTAL_BATCHES = math.ceil(TOTAL_RECORDS / PARSE_BATCH_SIZE)
print(f"Total batches (batch size {PARSE_BATCH_SIZE}): {TOTAL_BATCHES:,}")

In [ ]:
STANDARD_RESIDUES = set("ACDEFGHIKLMNPQRSTVWY")
RARE_RESIDUES = set("XBZJUO")  # misstyped tokens that appear in the dataset
VALID_RESIDUES = STANDARD_RESIDUES | RARE_RESIDUES

SPECIAL_TOKENS = [
    "[PAD]",  # padding
    "[UNK]",  # unknown token for out-of-vocabulary residues
    "[CLS]",  # start of sequence (classification token)
    "[SEP]",  # separator token (not used in single sequence tasks but reserved for potential future use)
    "[MASK]",  # mask token for masked language modeling
    "[UNKAA]",  # optional custom placeholder token for rare residues (if using replace_with_unk policy)
]

In [ ]:
records_total_count = protein_dataset.count_rows()
print(f"Records: {records_total_count:,}")

In [ ]:
def parquet_to_txt(parquet_path, output_txt_path, column="sequence", batch_size=100_000):
    dataset = ds.dataset(parquet_path, format="parquet")

    with open(output_txt_path, "w") as f:
        for batch in dataset.to_batches(columns=[column], batch_size=batch_size):
            arr = batch[column]

            # convert to python list (only this batch in memory)
            values = arr.to_pylist()

            for v in values:
                if v is not None:
                    f.write(v)
                    f.write("\n")

In [ ]:
str(DATA_PATH)

In [ ]:
parquet_to_txt(
    parquet_path=str(DATA_PATH),
    output_txt_path=str(OUTPUT_DIR / "corpus.txt"),
    column="sequence",
    batch_size=100_000,
)

In [ ]:
# !spm_train \
#   --input=corpus.txt \
#   --model_prefix=protein_spm \
#   --vocab_size=4096 \
#   --model_type=unigram \
#   --character_coverage=1.0 \
#   --split_digits=false \
# --normalization_rule_name=identity
#
!spm_train \
  --input='/home/ubuntu/projects/biodata/DNA-BERT/data/interim/protein_tokenization/corpus.txt' \
  --model_prefix=protein_spm \
  --vocab_size=8192 \
  --model_type=unigram \
  --character_coverage=1.0 \
  --split_digits=false \
  --normalization_rule_name=identity \
  --input_sentence_size=2000000 \
  --shuffle_input_sentence=true \
  --num_threads=4 \
  --max_sentence_length=4096